In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# === CONFIGURATION ===
DATASETS = {
    'rat': '/home/amenacer/Stage/base_de_donnees/rats/2-essaie-data11/aires_par_frame.csv',
    'souris': '/home/amenacer/Stage/base_de_donnees/rats/3-essaie-data12/aires_par_frame.csv',
    'mixte': '/home/amenacer/Stage/base_de_donnees/rats/4-essaie-data13/aires_par_frame.csv'
}
THRESHOLD = 0.10  # seuil de différence (10% de variation d'aire)

# === CHARGEMENT DES DONNEES ===
dfs = {name: pd.read_csv(path) for name, path in DATASETS.items()}

# === TRAITEMENT PAR RAT ===
rats = dfs['rat']['rat'].unique()
for rat in rats:
    plt.figure(figsize=(10, 5))
    for name, df in dfs.items():
        data = df[df['rat'] == rat].sort_values('frame')
        plt.plot(data['frame'], data['aire'], label=f'{name}')
    
    # Calcul de la différence relative (exemple rat vs souris)
    df_ref = dfs['rat'][dfs['rat']['rat'] == rat].sort_values('frame')
    for name, df_comp in dfs.items():
        if name == 'rat':
            continue
        df_c = df_comp[df_comp['rat'] == rat].sort_values('frame')
        # Différence relative
        diff = np.abs(df_ref['aire'].values - df_c['aire'].values) / np.maximum(df_ref['aire'].values, 1)
        anomalies = np.where(diff > THRESHOLD)[0]
        if len(anomalies) > 0:
            plt.scatter(df_ref['frame'].values[anomalies], df_c['aire'].values[anomalies],
                        marker='o', color='red', label=f'Δ>10% ({name})')
            print(f"Rat {rat} - {name}: Frames où la différence > {THRESHOLD*100}% : {df_ref['frame'].values[anomalies]}")
    
    plt.title(f'Comparaison des courbes Aire vs Frame — Rat {rat}')
    plt.xlabel('Frame')
    plt.ylabel('Aire segmentée')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages

# === CONFIGURATION ===
DATASETS = {
    'rat': '/home/amenacer/Stage/base_de_donnees/rats/2-essaie-data11/aires_par_frame.csv',
    'souris': '/home/amenacer/Stage/base_de_donnees/rats/3-essaie-data12/aires_par_frame.csv',
    'mixte': '/home/amenacer/Stage/base_de_donnees/rats/4-essaie-data13/aires_par_frame.csv'
}
THRESHOLD = 0.10  # seuil de différence (10% de variation d'aire)
PDF_FILENAME = 'comparaison_segmentation_rats.pdf'

# === CHARGEMENT DES DONNEES ===
dfs = {name: pd.read_csv(path) for name, path in DATASETS.items()}

# Pour stocker toutes les anomalies
anomalies_dict = {}

# === GENERATION DU PDF ===
with PdfPages(PDF_FILENAME) as pdf:
    rats = dfs['rat']['rat'].unique()
    for rat in rats:
        plt.figure(figsize=(10, 5))
        for name, df in dfs.items():
            data = df[df['rat'] == rat].sort_values('frame')
            plt.plot(data['frame'], data['aire'], label=f'{name}')
        
        # Calcul de la différence relative (par rapport à "rat")
        df_ref = dfs['rat'][dfs['rat']['rat'] == rat].sort_values('frame')
        for name, df_comp in dfs.items():
            if name == 'rat':
                continue
            df_c = df_comp[df_comp['rat'] == rat].sort_values('frame')
            diff = np.abs(df_ref['aire'].values - df_c['aire'].values) / np.maximum(df_ref['aire'].values, 1)
            anomalies = np.where(diff > THRESHOLD)[0]
            frames_anomalies = df_ref['frame'].values[anomalies]
            # Affichage sur le plot
            if len(anomalies) > 0:
                plt.scatter(df_ref['frame'].values[anomalies], df_c['aire'].values[anomalies],
                            marker='o', color='red', label=f'Δ>10% ({name})')
                # On sauvegarde les anomalies
                key = f"rat_{rat}_{name}"
                anomalies_dict[key] = list(frames_anomalies)
        plt.title(f'Comparaison Aire vs Frame — Rat {rat}')
        plt.xlabel('Frame')
        plt.ylabel('Aire segmentée')
        plt.legend()
        plt.tight_layout()
        pdf.savefig()
        plt.close()

    # === PAGE DE SYNTHÈSE DES ANOMALIES ===
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('off')
    y = 1
    ax.text(0, y, "Tableau des anomalies détectées (frames avec Δ > 10%)", fontsize=14, weight='bold')
    y -= 0.05
    for key, frames in anomalies_dict.items():
        ax.text(0, y, f"{key} : {frames}", fontsize=10)
        y -= 0.03
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close()

print(f"PDF généré : {PDF_FILENAME}")
